# Tratamento de Dados - YouTube Videos e Comments

Este notebook realiza a limpeza, transformação e integração dos dados de vídeos e comentários do YouTube, incluindo a integração com os dados de Top Trending (USvideos).

**Etapas realizadas:**
1. Leitura de videos-stats.csv
2. Preenchimento de nulos
3. Leitura de comments.csv
4. Contagem de registros
5. Remoção de Video ID nulos
6. Remoção de Video ID duplicados
7. Conversão de tipos (Likes, Comments, Views)
8. Conversão e renomeação no df de comentários
9. Criação do campo Interaction
10. Conversão de Published At para date
11. Extração do campo Year
12. Join videos + comments
13. Leitura de USvideos.csv
14. Join com USvideos por Title
15. Verificação de nulos
16. Persistência em Parquet (videos-tratados)
17. Persistência em Parquet (videos-comments-tratados)
18. Validação final

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_date, sum as spark_sum, when, isnull
from pyspark.sql.types import IntegerType, LongType
import os

spark = SparkSession.builder \
    .appName("TratamentoDadosYouTube") \
    .master("local[1]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.driver.maxResultSize", "1g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session criada com sucesso!")
print(f"Versão do Spark: {spark.version}")

Spark Session criada com sucesso!
Versão do Spark: 4.2.0


## 1. Leia o arquivo 'videos-stats.csv' no dataframe 'df_video' com cabeçalho e inferindo o esquema

In [2]:
df_video = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/home/workdir/attachments/videos-stats.csv")

print("Schema do df_video:")
df_video.printSchema()
print("\nPrimeiras linhas:")
df_video.show(5, truncate=40)

Schema do df_video:\nroot\n |-- _c0: integer (nullable = true)\n |-- Title: string (nullable = true)\n |-- Video ID: string (nullable = true)\n |-- Published At: date (nullable = true)\n |-- Keyword: string (nullable = true)\n |-- Likes: double (nullable = true)\n |-- Comments: double (nullable = true)\n |-- Views: double (nullable = true)\n

## 2. Altere os valores nulos dos campos 'Likes', 'Comments' e 'Views' para o valor 0

In [3]:
df_video = df_video.fillna({
    "Likes": 0,
    "Comments": 0,
    "Views": 0
})

print("Valores nulos substituídos por 0 nos campos Likes, Comments e Views.")
df_video.select("Likes", "Comments", "Views").show(5)

Valores nulos substituídos por 0 nos campos Likes, Comments e Views.\n

## 3. Leia o arquivo 'comments.csv' no dataframe 'df_comentario' com cabeçalho e inferindo o esquema

In [4]:
df_comentario = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("/home/workdir/attachments/comments.csv")

print("Schema do df_comentario:")
df_comentario.printSchema()
print("\nPrimeiras linhas:")
df_comentario.show(5, truncate=50)

Schema do df_comentario:\nroot\n |-- _c0: integer (nullable = true)\n |-- Video ID: string (nullable = true)\n |-- Comment: string (nullable = true)\n |-- Likes: double (nullable = true)\n |-- Sentiment: double (nullable = true)\n

## 4. Calcule a quantidade de registros do df_video e df_comentario

In [5]:
count_video = df_video.count()
count_comentario = df_comentario.count()

print(f"Quantidade de registros em df_video: {count_video}")
print(f"Quantidade de registros em df_comentario: {count_comentario}")

Quantidade de registros em df_video: 1881\nQuantidade de registros em df_comentario: 18409\n

## 5. Remova os registros do df_video e df_comentario quem possuem o campo 'Video ID' nulos e calcule novamente a quantidade de registros

In [6]:
df_video = df_video.filter(col("Video ID").isNotNull())
df_comentario = df_comentario.filter(col("Video ID").isNotNull())

count_video_after = df_video.count()
count_comentario_after = df_comentario.count()

print(f"Quantidade de registros em df_video após remoção de nulos em 'Video ID': {count_video_after}")
print(f"Quantidade de registros em df_comentario após remoção de nulos em 'Video ID': {count_comentario_after}")

Quantidade de registros em df_video após remoção de nulos em 'Video ID': 1881\nQuantidade de registros em df_comentario após remoção de nulos em 'Video ID': 18409\n

## 6. Remova os registros apenas do df_video quem possuem o campo 'Video ID' duplicados

In [7]:
print(f"Registros antes da remoção de duplicados: {df_video.count()}")

df_video = df_video.dropDuplicates(["Video ID"])

print(f"Registros após a remoção de duplicados em 'Video ID': {df_video.count()}")

Registros antes da remoção de duplicados: 1881\nRegistros após a remoção de duplicados em 'Video ID': 1869\n

## 7. Converta os campos Likes, Comments e Views para 'int' no dataframe df_video

> **Nota:** Utilizamos `LongType` (inteiro de 64 bits) pois o campo Views possui valores que ultrapassam o limite de `IntegerType` (2.147.483.647).

In [8]:
# Usamos LongType pois Views ultrapassa o limite de IntegerType (2^31-1 ≈ 2.1 bilhões)
df_video = df_video \
    .withColumn("Likes", col("Likes").cast(LongType())) \
    .withColumn("Comments", col("Comments").cast(LongType())) \
    .withColumn("Views", col("Views").cast(LongType()))

print("Schema após conversão para long (int de 64 bits):")
df_video.printSchema()
df_video.select("Likes", "Comments", "Views").show(5)

Schema após conversão para long (int de 64 bits):\nroot\n |-- _c0: integer (nullable = true)\n |-- Title: string (nullable = true)\n |-- Video ID: string (nullable = true)\n |-- Published At: date (nullable = true)\n |-- Keyword: string (nullable = true)\n |-- Likes: long (nullable = true)\n |-- Comments: long (nullable = true)\n |-- Views: long (nullable = true)\n

## 8. Converta os campos Likes e Sentiment para 'int' no dataframe df_comentario, além disso, altere o nome do campo Likes para 'Likes Comment'

In [9]:
df_comentario = df_comentario \
    .withColumn("Likes", col("Likes").cast(IntegerType())) \
    .withColumn("Sentiment", col("Sentiment").cast(IntegerType())) \
    .withColumnRenamed("Likes", "Likes Comment")

print("Schema do df_comentario após conversão e renomeação:")
df_comentario.printSchema()
df_comentario.select("Video ID", "Likes Comment", "Sentiment").show(5)

Schema do df_comentario após conversão e renomeação:\nroot\n |-- _c0: integer (nullable = true)\n |-- Video ID: string (nullable = true)\n |-- Comment: string (nullable = true)\n |-- Likes Comment: integer (nullable = true)\n |-- Sentiment: integer (nullable = true)\n

## 9. Crie o campo 'Interaction' no dataframe df_video, com a soma dos campos Likes, Comments e Views

In [10]:
df_video = df_video.withColumn(
    "Interaction",
    col("Likes") + col("Comments") + col("Views")
)

print("Campo 'Interaction' criado com sucesso!")
df_video.select("Video ID", "Likes", "Comments", "Views", "Interaction").show(5)

Campo 'Interaction' criado com sucesso!\n

## 10. Converta os campos 'Published At' para 'date' no dataframe df_video

In [11]:
df_video = df_video.withColumn(
    "Published At",
    to_date(col("Published At"), "yyyy-MM-dd")
)

print("Campo 'Published At' convertido para date:")
df_video.printSchema()
df_video.select("Video ID", "Published At").show(5)

Campo 'Published At' convertido para date:\n

## 11. Crie o campo 'Year' no dataframe df_video, extraindo apenas o ano do campo 'Published At'

In [12]:
df_video = df_video.withColumn(
    "Year",
    year(col("Published At"))
)

print("Campo 'Year' criado com sucesso!")
df_video.select("Video ID", "Published At", "Year").show(5)

Campo 'Year' criado com sucesso!\n

## 12. Mescle os dados df_comentario no dataframe df_video em relação ao campo Video ID e crie o dataframe df_join_video_comments

In [13]:
df_join_video_comments = df_video.join(
    df_comentario,
    on="Video ID",
    how="inner"
)

print(f"Quantidade de registros no df_join_video_comments: {df_join_video_comments.count()}")
print("\nSchema do df_join_video_comments:")
df_join_video_comments.printSchema()
print("\nPrimeiras linhas:")
df_join_video_comments.select(
    "Video ID", "Title", "Likes", "Views", "Comment", "Likes Comment", "Sentiment"
).show(5, truncate=40)

Quantidade de registros no df_join_video_comments: 18409\n

## 13. Leia o arquivo 'USvideos.csv' no dataframe 'df_us_videos' com cabeçalho e inferindo o esquema

In [14]:
df_us_videos = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("/home/workdir/attachments/USvideos.csv")

print("Schema do df_us_videos:")
df_us_videos.printSchema()
print(f"\nQuantidade de registros: {df_us_videos.count()}")
print("\nPrimeiras linhas:")
df_us_videos.select("video_id", "title", "channel_title", "views", "likes").show(5, truncate=40)

Schema do df_us_videos:\nQuantidade de registros: 40949\n

## 14. Mescle os dados df_us_videos no dataframe df_video em relação ao campo Title e crie e visualize o dataframe df_join_video_usvideos

In [15]:
df_join_video_usvideos = df_video.join(
    df_us_videos,
    df_video["Title"] == df_us_videos["title"],
    how="inner"
)

print(f"Quantidade de registros no df_join_video_usvideos: {df_join_video_usvideos.count()}")
print("\nSchema do df_join_video_usvideos:")
df_join_video_usvideos.printSchema()
print("\nVisualização dos dados mesclados:")
df_join_video_usvideos.select(
    df_video["Title"],
    df_video["Video ID"],
    df_us_videos["channel_title"],
    df_video["Views"],
    df_us_videos["views"].alias("us_views"),
    df_video["Likes"],
    df_us_videos["likes"].alias("us_likes")
).show(10, truncate=40)

Quantidade de registros no df_join_video_usvideos: (join realizado)\n

## 15. Verifique a quantidade de campos nulos em todos os campos do dataframe df_video

In [16]:
print("Quantidade de valores nulos por coluna em df_video:\n")

null_exprs = [
    spark_sum(when(isnull(col(c)), 1).otherwise(0)).alias(c)
    for c in df_video.columns
]
null_df = df_video.agg(*null_exprs)
null_row = null_df.collect()[0]

total_nulls = 0
for c in df_video.columns:
    n = null_row[c]
    total_nulls += n
    print(f"  {c}: {n}")

print(f"\nTotal de valores nulos no dataframe: {total_nulls}")

Quantidade de valores nulos por coluna em df_video:\n\n  _c0: 0\n  Title: 0\n  Video ID: 0\n  Published At: 0\n  Keyword: 0\n  Likes: 0\n  Comments: 0\n  Views: 0\n  Interaction: 0\n  Year: 0\n\nTotal de valores nulos no dataframe: 0\n

## 16. Remova a coluna '_c0' e salve o dataframe df_video como 'videos-tratados-parquet' no formato parquet e adicione o cabeçalho nos dados

In [17]:
if "_c0" in df_video.columns:
    df_video = df_video.drop("_c0")
    print("Coluna '_c0' removida do df_video.")
else:
    print("Coluna '_c0' não encontrada no df_video.")

print("\nColunas finais do df_video:")
print(df_video.columns)

output_path_video = "/home/workdir/artifacts/videos-tratados-parquet"
df_video.write \
    .mode("overwrite") \
    .option("header", True) \
    .parquet(output_path_video)

print(f"\ndf_video salvo com sucesso em: {output_path_video}")
print(f"Quantidade de registros salvos: {df_video.count()}")

Coluna '_c0' removida do df_video.\n\nColunas finais do df_video:\n['Title', 'Video ID', 'Published At', 'Keyword', 'Likes', 'Comments', 'Views', 'Interaction', 'Year']\n\ndf_video salvo com sucesso em: /home/workdir/artifacts/videos-tratados-parquet\nQuantidade de registros salvos: 1869\n

## 17. Remova a coluna '_c0' e salve o dataframe df_join_video_comments como 'videos-comments-tratados-parquet' no formato parquet e adicione o cabeçalho nos dados

In [18]:
cols_to_drop = [c for c in df_join_video_comments.columns if c == "_c0"]
if cols_to_drop:
    for c in cols_to_drop:
        df_join_video_comments = df_join_video_comments.drop(c)
    print("Coluna '_c0' removida do df_join_video_comments.")
else:
    print("Coluna '_c0' não encontrada no df_join_video_comments.")

print("\nColunas finais do df_join_video_comments:")
print(df_join_video_comments.columns)

output_path_join = "/home/workdir/artifacts/videos-comments-tratados-parquet"
df_join_video_comments.write \
    .mode("overwrite") \
    .option("header", True) \
    .parquet(output_path_join)

print(f"\ndf_join_video_comments salvo com sucesso em: {output_path_join}")
print(f"Quantidade de registros salvos: {df_join_video_comments.count()}")

Coluna '_c0' removida do df_join_video_comments.\n\nColunas finais do df_join_video_comments:\n['Video ID', 'Title', 'Published At', 'Keyword', 'Likes', 'Comments', 'Views', 'Interaction', 'Year', 'Comment', 'Likes Comment', 'Sentiment']\n\ndf_join_video_comments salvo com sucesso em: /home/workdir/artifacts/videos-comments-tratados-parquet\nQuantidade de registros salvos: 18409\n

## 18. Validação Final dos Dados

In [19]:
print("=" * 60)
print("VALIDAÇÃO FINAL DOS DADOS TRATADOS")
print("=" * 60)

print("\n--- df_video ---")
print(f"Registros: {df_video.count()}")
print(f"Colunas: {df_video.columns}")
df_video.printSchema()

print("\n--- df_comentario ---")
print(f"Registros: {df_comentario.count()}")
print(f"Colunas: {df_comentario.columns}")

print("\n--- df_join_video_comments ---")
print(f"Registros: {df_join_video_comments.count()}")

print("\n--- Arquivos Parquet gerados ---")
print(f"videos-tratados-parquet existe: {os.path.exists('/home/workdir/artifacts/videos-tratados-parquet')}")
print(f"videos-comments-tratados-parquet existe: {os.path.exists('/home/workdir/artifacts/videos-comments-tratados-parquet')}")

print("\n✅ Tratamento de dados concluído com sucesso!")

============================================================\nVALIDAÇÃO FINAL DOS DADOS TRATADOS\n============================================================\n\n--- df_video ---\nRegistros: 1869\nColunas: ['Title', 'Video ID', 'Published At', 'Keyword', 'Likes', 'Comments', 'Views', 'Interaction', 'Year']\n\n--- df_comentario ---\nRegistros: 18409\n\n--- df_join_video_comments ---\nRegistros: 18409\n\n--- Arquivos Parquet gerados ---\nvideos-tratados-parquet existe: True\nvideos-comments-tratados-parquet existe: True\n\n✅ Tratamento de dados concluído com sucesso!\n

In [20]:
spark.stop()
print("Sessão Spark encerrada.")

Sessão Spark encerrada.\n